# ERA5 day-of-year climatology — one-time precompute

**Purpose.** For each city in `pipeline/cities.csv`, compute a day-of-year (DOY) climatology of daily-mean 2-m temperature from ERA5 over the WMO standard normal period **1991–2020**, using a **15-day centered window** for smoothing.

**Why GEE.** The full ERA5 daily archive on a 0.25° grid is ~1.5 TB. Downloading and processing locally is impractical. Earth Engine has the data ingested and lets us reduce it in-cloud; we only download the small final result (~5 MB parquet).

**Output.** `era5_doy_clim.parquet` with rows of `(geonames_id, doy, mean_c, std_c, p95_c, p99_c)` — 366 DOYs × ~105 cities = ~38 k rows.

**Run cadence.** Run **once**, then again only when the WMO normal period rolls forward (next update: 2031, when the 2001–2030 normal becomes standard).

**Prerequisites.**
1. Sign up for Earth Engine at https://earthengine.google.com (free for nonprofit/research — fits ClimaHealth/GHHIN).
2. Run this notebook in **Google Colab** (the `earthengine-api` package is preinstalled). Your existing Colab Pro is fine; the free tier also works.
3. Authenticate Earth Engine on the first cell.

**License note.** ERA5 in GEE requires the Copernicus attribution: *"Generated using Copernicus Climate Change Service information [year]"*. This is included in the output metadata and surfaced in the frontend colophon.

In [ ]:
# Cell 1 — install + auth
!pip install -q earthengine-api pandas pyarrow

import ee
ee.Authenticate()        # opens a browser for OAuth
ee.Initialize(project='YOUR-EE-PROJECT-ID')   # ← replace with your Cloud project

In [ ]:
# Cell 2 — load city list
import pandas as pd
from google.colab import files

# Option A: upload pipeline/cities.csv via the Colab files panel
# Option B: read directly from the public GitHub repo once it's live:
# cities = pd.read_csv('https://raw.githubusercontent.com/<user>/<repo>/main/pipeline/cities.csv')

uploaded = files.upload()
cities = pd.read_csv(next(iter(uploaded.keys())))
print(f'Loaded {len(cities)} cities')
cities.head()

In [ ]:
# Cell 3 — config
START_YEAR = 1991
END_YEAR = 2020
WINDOW_DAYS = 15           # 15-day centered window
HALF_WINDOW = WINDOW_DAYS // 2

# ERA5/DAILY band: 'mean_2m_air_temperature' is daily mean in Kelvin.
# We compute climatology of daily-mean temp; the live GFS sample is a
# point-in-time analysis. They're not identical quantities, but they share
# the same DOY structure and the percentile metric is what matters.
ERA5 = ee.ImageCollection('ECMWF/ERA5/DAILY').select('mean_2m_air_temperature')

# Build city FeatureCollection
feats = [
    ee.Feature(ee.Geometry.Point([row.lon, row.lat]),
               {'geonames_id': int(row.geonames_id), 'name': row['name']})
    for row in cities.itertuples()
]
city_fc = ee.FeatureCollection(feats)

In [ ]:
# Cell 4 — sample full daily series at city points (1991-2020)
# This is the heavy step. ERA5 daily mean × 30 years × ~105 cities is
# 30 × 365.25 × 105 ≈ 1.15 M samples. GEE handles this comfortably as a
# server-side reduce; the local export is the only constrained step.
#
# Strategy: sample one year at a time, accumulate into a single FeatureCollection,
# export to Drive as CSV, then post-process locally.

def sample_year(year):
    start = f'{year}-01-01'
    end = f'{year + 1}-01-01'
    ic = ERA5.filterDate(start, end)
    # For each image, sample at all city points and tag with the date
    def per_image(img):
        date = img.date().format('YYYY-MM-dd')
        sampled = img.reduceRegions(
            collection=city_fc,
            reducer=ee.Reducer.first(),
            scale=27830  # ERA5 native ~27.8 km
        )
        return sampled.map(lambda f: f.set('date', date))
    return ic.map(per_image).flatten()

# Kick off one export per year — keeps each export small and parallelizable
tasks = []
for year in range(START_YEAR, END_YEAR + 1):
    fc = sample_year(year)
    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=f'era5_samples_{year}',
        folder='heatmap_climatology',
        fileNamePrefix=f'era5_samples_{year}',
        fileFormat='CSV',
        selectors=['geonames_id', 'date', 'first']
    )
    task.start()
    tasks.append((year, task))
    print(f'Started export for {year}: task id {task.id}')

Wait for the 30 export tasks to finish — check the Earth Engine Tasks panel at https://code.earthengine.google.com/tasks. Each year takes a few minutes; in parallel they should all finish in well under an hour. Files land in `Google Drive / heatmap_climatology / era5_samples_<year>.csv`.

Once they're done, mount Drive and run the next cells locally.

In [ ]:
# Cell 5 — pull the per-year CSVs from Drive and stack
from google.colab import drive
drive.mount('/content/drive')

import glob
import pandas as pd

paths = sorted(glob.glob('/content/drive/MyDrive/heatmap_climatology/era5_samples_*.csv'))
print(f'Found {len(paths)} year files')
df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
df = df.rename(columns={'first': 'temp_k'})
df['temp_c'] = df['temp_k'] - 273.15
df['date'] = pd.to_datetime(df['date'])
df['doy'] = df['date'].dt.dayofyear
# Treat Feb 29 (DOY 60 in leap years) as DOY 60 mapping to a synthetic 'late-Feb' bucket;
# in non-leap years the original DOY 60 is March 1. Standard practice: collapse Feb 29
# samples into DOY 59 to avoid sparse-DOY artifacts.
df.loc[(df['date'].dt.month == 2) & (df['date'].dt.day == 29), 'doy'] = 59
print(df.head())
print(f'Total samples: {len(df):,}')

In [ ]:
# Cell 6 — compute the 15-day centered DOY climatology per city
import numpy as np

def doy_window(doy, half=HALF_WINDOW):
    """Return the set of DOYs in a centered window, wrapping around year-end."""
    return [((doy - 1 + d) % 365) + 1 for d in range(-half, half + 1)]

# Pre-bucket by (city, doy) for fast windowed queries
buckets = df.groupby(['geonames_id', 'doy'])['temp_c'].apply(list).to_dict()

rows = []
all_cities = df['geonames_id'].unique()
for gid in all_cities:
    for doy in range(1, 366):
        # Pull all samples in the 15-day window
        window_temps = []
        for d in doy_window(doy):
            window_temps.extend(buckets.get((gid, d), []))
        if len(window_temps) < 100:    # 30 yrs × 15 days = ~450 expected; flag if low
            print(f'WARN: gid={gid} doy={doy} has only {len(window_temps)} samples')
        arr = np.array(window_temps)
        rows.append({
            'geonames_id': gid,
            'doy': doy,
            'mean_c': float(arr.mean()),
            'std_c':  float(arr.std(ddof=1)),
            'p95_c':  float(np.percentile(arr, 95)),
            'p99_c':  float(np.percentile(arr, 99)),
            'n':      int(len(arr))
        })

clim = pd.DataFrame(rows)
print(clim.describe())
clim.head()

In [ ]:
# Cell 7 — sanity-check a couple of cities
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for ax, gid in zip(axes.flat, [3530597, 1275339, 2147714, 524901]):
    sub = clim[clim['geonames_id'] == gid].sort_values('doy')
    name = cities.loc[cities['geonames_id'] == gid, 'name'].iloc[0]
    ax.fill_between(sub['doy'], sub['mean_c'] - sub['std_c'], sub['mean_c'] + sub['std_c'],
                    alpha=0.25, label='±1σ')
    ax.plot(sub['doy'], sub['mean_c'], lw=1.5, label='mean')
    ax.plot(sub['doy'], sub['p95_c'], lw=1, ls='--', label='p95')
    ax.set_title(name); ax.set_ylabel('°C'); ax.grid(alpha=0.3)
axes[1, 0].set_xlabel('Day of year')
axes[1, 1].set_xlabel('Day of year')
axes[0, 0].legend(loc='best', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8 — write parquet + download
out_path = '/content/era5_doy_clim.parquet'
clim.to_parquet(out_path, index=False)
print(f'Wrote {out_path} — {clim.shape[0]:,} rows, {clim.memory_usage(deep=True).sum() / 1024:.1f} KB')

from google.colab import files
files.download(out_path)
# Then commit this file to your repo at climatology/era5_doy_clim.parquet